# Surface biharmonic operator and convergence

This notebook solves a manufactured surface biharmonic problem on the unit
sphere and measures p-refinement convergence.

In [ ]:
from pathlib import Path
import sys
import time

import numpy as np

project = Path.cwd().resolve()
if project.name == 'notebooks':
    project = project.parent
sys.path.insert(0, str(project))

import pysurfacefun as psf

output_dir = project / 'notebook_outputs'
output_dir.mkdir(exist_ok=True)

## Model problem

On the unit sphere, spherical harmonics satisfy

$$
\Delta_\Gamma Y_\ell^m = -\ell(\ell+1)Y_\ell^m.
$$

For the biharmonic problem

$$
\Delta_\Gamma^2 u = f,
$$

we use the manufactured solution $u=Y_\ell^m$.  With
$\lambda=\ell(\ell+1)$, the forcing is

$$
f = \lambda^2 Y_\ell^m.
$$

The fourth-order equation is solved by two second-order solves:

$$
\Delta_\Gamma w = f,
\qquad
\Delta_\Gamma u = w.
$$

Since the surface is closed, each Laplace--Beltrami solve is rank deficient;
the right-hand sides are projected to mean zero and the computed solutions are
reported with zero mean.

In [ ]:
test_modes = [
    (4, 3, r'$Y_4^3$'),
    (14, 10, r'$Y_{14}^{10}$'),
    (20, 13, r'$Y_{20}^{13}$'),
]

p_values = np.arange(4, 21)       # polynomial degrees
nref = 3                          # fixed patch refinement level
rho_reference = 9.3               # reference spectral rate shown on the slide

In [ ]:
def real_zero_mean(field):
    return psf.real(field.remove_mean())


def exact_solution(dom, ell, m):
    return real_zero_mean(psf.surfacefun(
        lambda x, y, z: psf.real_spherical_harmonic(ell, m, x, y, z),
        dom,
    ))


def solve_rankdef_laplace(rhs):
    rhs0 = real_zero_mean(rhs)
    L = psf.surfaceop(rhs0.domain, {'lap': 1.0}, rhs0)
    L.rankdef = True
    return real_zero_mean(L.solve())


def solve_biharmonic_mode(dom, ell, m):
    u_exact = exact_solution(dom, ell, m)
    lam = ell * (ell + 1)
    f = (lam * lam) * u_exact

    w_h = solve_rankdef_laplace(f)
    u_h = solve_rankdef_laplace(w_h)

    relerr = psf.norm(u_h - u_exact, 'inf') / psf.norm(u_exact, 'inf')
    return u_exact, f, w_h, u_h, relerr

## One solve

The following cell computes one representative solution.  It also exports a VTU
file if `meshio` is installed.

In [ ]:
ell, m, label = test_modes[-1]
dom = psf.sphere(n=17, nref=nref)

start = time.perf_counter()
u_exact, f, w_h, u_h, relerr = solve_biharmonic_mode(dom, ell, m)
elapsed = time.perf_counter() - start

print(f'mode                 : {label}')
print(f'patches              : {dom.npatches}')
print(f'polynomial degree    : {dom.n - 1}')
print(f'relative L_inf error : {relerr:.3e}')
print(f'solve time           : {elapsed:.3f} s')

In [ ]:
try:
    psf.plot_surface(u_h, title='Surface biharmonic solution', colorbar=False)
except Exception as exc:
    print('surface plot skipped:', exc)

## p-refinement convergence

For each polynomial degree, the patch layout is fixed and the local polynomial
order is increased.  Each data point uses two Laplace--Beltrami solves.

In [ ]:
rows = []
errors = {label: [] for _, _, label in test_modes}

for p in p_values:
    n = int(p + 1)
    dom_p = psf.sphere(n=n, nref=nref)
    print(f'p = {p:2d}, patches = {dom_p.npatches:3d}')

    for ell, m, label in test_modes:
        start = time.perf_counter()
        _, _, _, sol, err = solve_biharmonic_mode(dom_p, ell, m)
        elapsed = time.perf_counter() - start
        errors[label].append(err)
        rows.append((p, n, dom_p.npatches, ell, m, err, elapsed))
        print(f'  {label:12s} error = {err:.3e}, time = {elapsed:.3f} s')

# results = np.asarray(rows, dtype=float)
# np.savetxt(
#     output_dir / 'surface_biharmonic_convergence.txt',
#     results,
#     header='degree n npatches ell m relative_Linf_error solve_time_seconds',
#     fmt=['%d', '%d', '%d', '%d', '%d', '%.16e', '%.8e'],
# )

In [ ]:
try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(7.2, 4.8))
    styles = {
        r'$Y_4^3$': ('s-', 'red'),
        r'$Y_{14}^{10}$': ('o-', '#2f7d2a'),
        r'$Y_{20}^{13}$': ('d-', 'blue'),
    }

    for _, _, label in test_modes:
        marker, color = styles[label]
        ax.semilogy(
            p_values,
            errors[label],
            marker,
            color=color,
            linewidth=2.2,
            markersize=6.0,
            markerfacecolor='none' if label == r'$Y_{14}^{10}$' else color,
            markeredgewidth=2.0,
            label=label,
        )

    anchor_label = r'$Y_{14}^{10}$'
    anchor_index = min(4, len(p_values) - 1)
    anchor_p = p_values[anchor_index]
    anchor_y = errors[anchor_label][anchor_index]
    reference = anchor_y * rho_reference ** (-(p_values - anchor_p))
    ax.semilogy(p_values, reference, 'k--', linewidth=2.0, label=fr'$\rho^{{-p}}$, $\rho\approx {rho_reference}$')

    ax.set_xlabel('Polynomial degree')
    ax.set_ylabel(r'$\|u-u_h\|_{L_\infty}$')
    ax.set_ylim(1e-15, 1e1)
    ax.grid(True, which='major', alpha=0.30)
    ax.grid(True, which='minor', alpha=0.12)
    ax.legend(loc='lower left', frameon=True, fancybox=False, edgecolor='black')
    fig.tight_layout()
    fig.savefig(output_dir / 'surface_biharmonic_convergence.png', dpi=300)
    fig.savefig(output_dir / 'surface_biharmonic_convergence.pdf')
    plt.show()
except Exception as exc:
    print('convergence plot skipped:', exc)

## Output files

The notebook writes:

- `notebook_outputs/surface_biharmonic_convergence.txt`
- `notebook_outputs/surface_biharmonic_convergence.png`
- `notebook_outputs/surface_biharmonic_convergence.pdf`
- `notebook_outputs/surface_biharmonic_sphere_solution.vtu`, if `meshio` is installed